# Deploy AgentCore Memory for Betting & Gaming Demo

Creates an AgentCore Memory resource with a long-term strategy:
- **User Preference**  - captures preferences (e.g. "prefers football betting")

Short-term memory is included by default for within-session conversation persistence.

<hr>

In [ ]:
!pip install bedrock-agentcore strands-agents strands-agents-tools --quiet

## Step 1: Create Memory Resource
<hr>

In [ ]:
import time
import logging
import boto3
from datetime import datetime
from botocore.exceptions import ClientError
from bedrock_agentcore.memory import MemoryClient

REGION = "us-east-1"
EXECUTION_ROLE_ARN = "arn:aws:iam::869795389844:role/AgentCoreDemoRole"

# Set this to an existing memory ID to skip creation, or leave as None
memory_id = "agentic_ai_demo-AwrsGjE0oe"

client = MemoryClient(region_name=REGION)
control_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

# If memory_id is provided, verify it exists
if memory_id:
    try:
        info = control_client.get_memory(memoryId=memory_id)
        print(f"Memory already exists: {memory_id} (status: {info.get('status')})")
    except Exception as e:
        print(f"Memory ID '{memory_id}' not found: {e}")
        memory_id = None

# Create new memory if needed (using boto3 for full control over API fields)
if not memory_id:
    try:
        print("Creating new memory...")
        resp = control_client.create_memory(
            name="agentic_ai_demoa",
            description="Agentic AI Demo",
            memoryExecutionRoleArn=EXECUTION_ROLE_ARN,
            eventExpiryDuration=90,
            memoryStrategies=[
                {
                    "userPreferenceMemoryStrategy": {
                        "name": "PreferenceLearner",
                        "description": "Captures user preferences across sessions",
                        "namespaceTemplates": ["/preferences/user/{actorId}/"]
                    }
                },
            ],
        )
        memory_id = resp["memory"]["id"]
        print(f"Memory created: {memory_id} (status: {resp['memory']['status']})")

        # Wait for ACTIVE status
        print("Waiting for memory to become ACTIVE...")
        for _ in range(30):
            info = control_client.get_memory(memoryId=memory_id)
            status = info["memory"]["status"]
            if status == "ACTIVE":
                print(f"Memory is ACTIVE")
                break
            time.sleep(10)
        else:
            print(f"Warning: memory status is {status} after waiting")

    except ClientError as e:
        if "already exists" in str(e):
            memories = control_client.list_memories()
            for m in memories.get("memories", []):
                if m.get("name") == "agentic_ai_demo":
                    memory_id = m["id"]
                    break
            print(f"Memory already exists. Using: {memory_id}")
        else:
            raise

print(f"\nMemory ID: {memory_id}")

## Step 2: Verify Memory Resource
<hr>

In [ ]:
info = control_client.get_memory(memoryId=memory_id)["memory"]

print(f"Memory ID: {info.get('id')}")
print(f"Name: {info.get('name')}")
print(f"Status: {info.get('status')}")

strategies = info.get('strategies', [])
print(f"\nStrategies ({len(strategies)}):")
for s in strategies:
    print(f"  - {s.get('name')} (type: {s.get('type')}, id: {s.get('strategyId')})")
    print(f"    namespaces: {s.get('namespaceTemplates', [])}")

## Step 3: Hydrate Short-Term Memory

Save a sample conversation directly to short-term memory using `create_event`.
This uses proper conversational payloads that trigger long-term extraction automatically.
<hr>

In [ ]:
actor_id = "cust-003"
session_id = "session-2026042511"

# Namespace for preference retrieval (matches the PreferenceLearner template)
preference_namespace = f"/preferences/user/{actor_id}/"

print(f"Actor ID:  {actor_id}")
print(f"Session ID: {session_id}")
print(f"Namespace:  {preference_namespace}")

In [ ]:
# Sample conversation  - betting & gaming context
previous_messages = [
    ("Hi, my name is John and I'm a big football fan.", "USER"),
    ("Hi John! Great to have you here. Are you interested in football betting or just following the games?", "ASSISTANT"),
    ("I love betting on Premier League matches, especially Arsenal games.", "USER"),
    ("Arsenal is a great team to follow! Do you usually bet on match results, or do you prefer other markets like over/under or both teams to score?", "ASSISTANT"),
    ("I mostly bet on match results  - home win, draw, away win. I usually keep my stakes around $20-50.", "USER"),
    ("That's a solid approach. Match result bets are straightforward and fun. With $20-50 stakes, you're keeping it responsible too. Would you like me to check the upcoming Arsenal fixtures and odds?", "ASSISTANT"),
    ("Yes please! Also, I prefer to bet on weekends when I can watch the games live.", "USER"),
    ("Perfect! I'll focus on weekend Premier League matches for you. Watching live while having a bet on makes it much more exciting!", "ASSISTANT"),
]

print(f"Hydrating short-term memory with {len(previous_messages)} messages...")

event = client.create_event(
    memory_id=memory_id,
    actor_id=actor_id,
    session_id=session_id,
    messages=previous_messages,
)
print(f"Event created: {event.get('eventId')}")

#### Verify Short-Term Memory Events

In [ ]:
events = client.list_events(
    memory_id=memory_id,
    actor_id=actor_id,
    session_id=session_id,
    max_results=10,
)
print(f"Found {len(events)} event(s) in session '{session_id}'")
for evt in events:
    print(f"  Event: {evt.get('eventId')}  Time: {evt.get('eventTimestamp')}")

## Step 4: Retrieve Long-Term Memories

Wait ~30 seconds for the extraction strategies to process the conversation,
then retrieve the extracted long-term memories using semantic search.
<hr>

In [ ]:
# Wait for extraction to process
#...

# Retrieve preferences using semantic search
try:
    preferences = client.retrieve_memories(
        memory_id=memory_id,
        namespace=preference_namespace,
        query="betting preferences",
        top_k=5,
    )

    if preferences:
        print(f"Retrieved {len(preferences)} preference record(s):\n")
        for i, record in enumerate(preferences):
            print(f"  Memory {i+1}: {record.get('content', 'N/A')}")
    else:
        print("No preference records found yet.")
        print("Extraction may still be processing  - re-run this cell in 30 seconds.")

except Exception as e:
    print(f"Error retrieving memories: {e}")

## Step 5: Test Agent with Memory Tool

Create a Strands agent that uses `AgentCoreMemoryToolProvider` to store and retrieve
memories as a tool  - enabling the agent to recall user preferences across sessions.
<hr>

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands_tools.agent_core_memory import AgentCoreMemoryToolProvider

model = BedrockModel(model_id="global.anthropic.claude-opus-4-6-v1")

system_prompt = f"""You are a helpful betting and customer support assistant for a sports betting platform.

You have access to a Memory tool that enables you to:
- Store user preferences (favorite sports, teams, betting style, stake preferences)
- Retrieve previously stored information to personalize recommendations

Always check memory first before asking the user for information they may have already shared.
"""

# Create memory tool provider  - connects the agent to long-term memory
provider = AgentCoreMemoryToolProvider(
    memory_id=memory_id,
    actor_id=actor_id,
    session_id=session_id,
    namespace=preference_namespace,
    region=REGION

)

agent = Agent(
    model=model,
    tools=provider.tools,
    system_prompt=system_prompt,
)

print(f"Agent created with tools: {agent.tool_names}")

In [ ]:
# The agent should retrieve memories and give personalized recommendations
response = agent("What bets would you recommend for me this weekend?")

## Step 6: Inspect All Memory Records
<hr>

In [68]:
import boto3
import json

data_client = boto3.client('bedrock-agentcore', region_name=REGION)

print("--- Long-term memory records ---")
namespace_paths = [
    f"/preferences/user/{actor_id}/",
    f"/preferences/actor/{actor_id}/",  # legacy namespace
]

all_records = []
for ns_path in namespace_paths:
    print(f"\n  Namespace: {ns_path}")
    try:
        records = data_client.list_memory_records(memoryId=memory_id, namespacePath=ns_path)
        for r in records.get("memoryRecordSummaries", []):
            print(f"    [{r.get('namespaces', [''])[0]}] {r.get('content', {}).get('text', 'N/A')[:250]}")
            all_records.append(r)
        if not records.get("memoryRecordSummaries"):
            print("    No records yet.")
    except Exception as e:
        print(f"    Error: {e}")

# Pretty-print extracted preferences
if all_records:
    print(f"\n{'='*60}")
    print(f"  Extracted Preferences for actor: {actor_id}")
    print(f"{'='*60}")
    for i, r in enumerate(all_records, 1):
        raw = r.get('content', {}).get('text', '')
        try:
            pref = json.loads(raw)
            print(f"\n  {i}. {pref.get('preference', raw)}")
            if pref.get('categories'):
                print(f"     Categories: {', '.join(pref['categories'])}")
            if pref.get('context'):
                print(f"     Context: {pref['context']}")
        except (json.JSONDecodeError, TypeError):
            print(f"\n  {i}. {raw[:150]}")
else:
    print("\nNo records found. Extraction may still be processing.")
    print("Wait 30-60 seconds and re-run this cell.")

--- Long-term memory records ---

  Namespace: /preferences/user/cust-003/
    [/preferences/user/cust-003/] {"context":"The user explicitly stated that they like betting in NBA games.","preference":"Likes betting on NBA games","categories":["sports","betting","basketball","NBA"]}
    [/preferences/user/cust-003/] {"context":"The user explicitly mentioned being a basketball fan.","preference":"Basketball fan","categories":["sports","basketball","entertainment"]}
    [/preferences/user/cust-003/] {"context":"The user explicitly stated that their favorite player is Lebron.","preference":"Favorite player is LeBron James","categories":["sports","basketball","NBA","athletes"]}

  Namespace: /preferences/actor/cust-003/
    No records yet.

  Extracted Preferences for actor: cust-003

  1. Likes betting on NBA games
     Categories: sports, betting, basketball, NBA
     Context: The user explicitly stated that they like betting in NBA games.

  2. Basketball fan
     Categories: sports, bask

## Step 7: Delete All Memory Records
Deletes all short-term (session events) and long-term memory records for the test actor.
<hr>

In [ ]:
import boto3

data_client = boto3.client('bedrock-agentcore', region_name=REGION)
deleted_lt = 0
deleted_st = 0

# --- Delete long-term memory records ---
print("=== Deleting long-term memory records ===")
namespace_paths = [
    f"/preferences/user/{actor_id}/",
    f"/preferences/actor/{actor_id}/",  # legacy namespace
]

for ns_path in namespace_paths:
    try:
        records = data_client.list_memory_records(memoryId=memory_id, namespacePath=ns_path)
        record_ids = [r['memoryRecordId'] for r in records.get('memoryRecordSummaries', [])]
        if record_ids:
            resp = data_client.batch_delete_memory_records(
                memoryId=memory_id,
                records=[{'memoryRecordId': rid} for rid in record_ids],
            )
            deleted_lt += len(resp.get('successfulRecords', []))
            failed = resp.get('failedRecords', [])
            if failed:
                print(f"  {ns_path}: {len(failed)} failed to delete")
    except Exception as e:
        print(f"  {ns_path}: {e}")

print(f"  Deleted {deleted_lt} long-term records")

# --- Delete short-term memory (session events) ---
# Delete events for all actor ID variants (the session manager creates events
# under the raw actor_id, plus prefixed session_ and agent_ variants)
print("\n=== Deleting short-term memory events ===")
actor_variants = [actor_id, f"session_{session_id}", "agent_default"]
for aid in actor_variants:
    try:
        sessions = data_client.list_sessions(memoryId=memory_id, actorId=aid)
        for s in sessions.get('sessionSummaries', []):
            sid = s['sessionId']
            events = data_client.list_events(
                memoryId=memory_id, sessionId=sid, actorId=aid,
            )
            for evt in events.get('events', []):
                data_client.delete_event(
                    memoryId=memory_id, sessionId=sid,
                    eventId=evt['eventId'], actorId=aid,
                )
                deleted_st += 1
            if events.get('events'):
                print(f"  [{aid}] Session {sid}: deleted {len(events['events'])} events")
    except Exception as e:
        print(f"  [{aid}] Error: {e}")

print(f"  Deleted {deleted_st} short-term events total")
print(f"\nDone. Cleared {deleted_lt} long-term records + {deleted_st} short-term events.")

## Clean Up (run only when done with the demo)
<hr>

In [ ]:
# # Uncomment to delete the memory resource entirely
# client.delete_memory_and_wait(
#     memory_id=memory_id,
#     max_wait=300,
#     poll_interval=10,
# )
# print(f"Deleted memory: {memory_id}")